<a href="https://colab.research.google.com/github/abhirupchak/Wheat-Yield-Regression-Analysis-FAOSTAT/blob/main/naive_bayes_mushroom.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.naive_bayes import CategoricalNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report
)

# Upload dataset
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# 1. Load dataset
df = pd.read_csv(file_name)

# 2. Separate features and target
X = df.drop(columns=["Class", "SampleID"])
y = df["Class"].map({"edible": 1, "poisonous": 0})

# 3. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 4. Naive Bayes pipeline
model = Pipeline(
    steps=[
        ("encoder", OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        )),
        ("classifier", CategoricalNB())
    ]
)

# 5. Train model
model.fit(X_train, y_train)

# 6. Predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# 7. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

TN, FP, FN, TP = cm.ravel()

# 8. Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
specificity = TN / (TN + FP)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
error = 1 - accuracy

# 9. Evaluation results
results = pd.DataFrame({
    "Measure": [
        "Accuracy",
        "TP",
        "TN",
        "FP",
        "FN",
        "Error",
        "Recall",
        "Specificity",
        "F1 Score",
        "AUC"
    ],
    "Score": [
        accuracy,
        TP,
        TN,
        FP,
        FN,
        error,
        recall,
        specificity,
        f1,
        auc
    ]
})

print("DATASET SHAPE:", df.shape)

print("\nEVALUATION MEASURES")
display(results)

# 10. Confusion Matrix
cm_df = pd.DataFrame(
    cm,
    index=["Actual Poisonous", "Actual Edible"],
    columns=["Predicted Poisonous", "Predicted Edible"]
)

print("\nCONFUSION MATRIX")
display(cm_df)

# 11. Classification Report
print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Poisonous", "Edible"]
))

Saving 11_mushroom_edibility.csv to 11_mushroom_edibility.csv
DATASET SHAPE: (600, 11)

EVALUATION MEASURES


,Measure,Score
0,Accuracy,0.800000
1,TP,33.000000
2,TN,63.000000
3,FP,6.000000
4,FN,18.000000
5,Error,0.200000
6,Recall,0.647059
7,Specificity,0.913043
8,F1 Score,0.733333
9,AUC,0.875959



CONFUSION MATRIX


,Predicted Poisonous,Predicted Edible
Actual Poisonous,63,6
Actual Edible,18,33



CLASSIFICATION REPORT
              precision    recall  f1-score   support

   Poisonous       0.78      0.91      0.84        69
      Edible       0.85      0.65      0.73        51

    accuracy                           0.80       120
   macro avg       0.81      0.78      0.79       120
weighted avg       0.81      0.80      0.79       120

